## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Authenticate with HuggingFace to access gated models.
# 1. Create a token at https://huggingface.co/settings/tokens (Read access is enough).
# 2. In Colab: click the 🔑 Secrets icon in the left sidebar → add secret named HF_TOKEN.
# 3. Request access at https://huggingface.co/openbmb/MiniCPM-V-2_6 (granted instantly).
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Pin transformers to 4.44.2: MiniCPM-V-2.6's custom MiniCPMV class does not
# implement all_tied_weights_keys, a property added to PreTrainedModel in 4.45+.
# timm is required by the SigLIP vision encoder used in MiniCPM-V-2.6.
%pip install -q "transformers==4.44.2" accelerate bitsandbytes timm Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.2 MB/s eta 0:00:00:00:0100:01


In [3]:
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

In [4]:
import torch

In [ ]:
import sys
import types
import importlib.util
from pathlib import Path
from PIL import Image

# MiniCPM-V-2.6's remote modeling file has an unconditional `import flash_attn`
# at module level. flash-attn requires Ampere+ (sm80+) to compile; T4 is Turing (sm75).
# We inject a proper stub module (types.ModuleType + valid __spec__) so Python's
# import machinery is satisfied. MagicMock fails because it leaves __spec__ unset.
class _DummyModule(types.ModuleType):
    """Stub that silently accepts any attribute access or call."""
    def __getattr__(self, name):
        return lambda *a, **kw: None

for _name in ('flash_attn', 'flash_attn.flash_attn_interface', 'flash_attn.bert_padding'):
    if _name not in sys.modules:
        _mod = _DummyModule(_name)
        _mod.__spec__ = importlib.util.spec_from_loader(_name, loader=None)
        sys.modules[_name] = _mod

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'openbmb/MiniCPM-V-2_6'

# MiniCPM-V-2.6 is 8B params → ~4 GB at 4-bit, feasible on T4's 15 GB.
qc = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=qc,
    device_map='auto',
    torch_dtype=torch.float16,
    attn_implementation='eager',
).eval()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/openbmb/MiniCPM-V-2_6.
401 Client Error. (Request ID: Root=1-69ec208f-314bca877d4061fe4eb5f50b;6903cd63-6602-4f15-b852-f9a85634924a)

Cannot access gated repo for url https://huggingface.co/openbmb/MiniCPM-V-2_6/resolve/main/config.json.
Access to model openbmb/MiniCPM-V-2_6 is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
IMAGE_FILE = WORKING_DIR / 'images/pineda1/pineda1_page_3.png'

## Inference

In [ ]:
import time

image_stem   = IMAGE_FILE.stem
image_folder = IMAGE_FILE.parent.name

image = Image.open(IMAGE_FILE).convert('RGB')

question = (
    'Convert the document to plain text, as close to the original as possible '
    '(including typos, print errors, and original grammar and spelling). '
    'Do not add any formatting, markdown, or annotations.'
)

# MiniCPM-V-2.6 expects PIL Images directly inside the content list.
msgs = [{'role': 'user', 'content': [image, question]}]

t0 = time.time()
with torch.no_grad():
    transcription = model.chat(
        image=None,
        msgs=msgs,
        tokenizer=tokenizer,
        max_new_tokens=4096,
        sampling=False,   # greedy — avoids multinomial NaN issues on T4 fp16
    )
elapsed = time.time() - t0

print(f'Done in {elapsed:.1f}s')
print(transcription)

### Saving the output

In [ ]:
# Transcription → transcriptions/MiniCPM-V-2_6/<stem>.md
transcription_out = WORKING_DIR / 'transcriptions/MiniCPM-V-2_6'
transcription_out.mkdir(parents=True, exist_ok=True)
(transcription_out / f'{image_stem}.md').write_text(transcription, encoding='utf-8')
print(f'Saved: transcriptions/MiniCPM-V-2_6/{image_stem}.md')